# Batch enqueue from `temp/` via `frontman`

This notebook recursively scans all files in `temp/` and sends each recording to `frontman` (`/add_job`) so the normal pipeline (ASR, optional diarization, post-processing) handles it.

Set the batch-level parameters once, then run all cells.

In [1]:
from pathlib import Path
from datetime import datetime
import requests
import json
import time

# Batch settings
TEMP_ROOT = Path("temp")
FRONTMAN_BASE_URL = "http://127.0.0.1:8005"
SOURCE_NAME = "manual"

# Use Whisper language code or "auto" (for example: "ru", "en", "auto")
LANGUAGE = "ru"

# Diarization settings for the whole batch
ENABLE_DIARIZATION = True
N_SPEAKERS = 2

# Queue behavior
BATCH_SIZE = 5
POLL_INTERVAL_SEC = 10
FINAL_STATUSES = {"transcribed", "error"}

# Keep only media files you want to process
ALLOWED_EXTENSIONS = {
    ".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma", ".mp4", ".webm"
}

print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Temp root: {TEMP_ROOT.resolve()}")
print(f"Frontman URL: {FRONTMAN_BASE_URL}")
print(f"Language: {LANGUAGE}, diarization: {ENABLE_DIARIZATION}, n_speakers: {N_SPEAKERS}")
print(f"Batch size: {BATCH_SIZE}, poll interval: {POLL_INTERVAL_SEC}s")

Started: 2026-05-07 16:08:45
Temp root: C:\Users\Alex\whisper_asr_implementation\temp
Frontman URL: http://127.0.0.1:8005
Language: ru, diarization: True, n_speakers: 2
Batch size: 5, poll interval: 10s


In [2]:
def collect_media_files(root, allowed_extensions):
    root = root.resolve()
    if not root.exists():
        raise FileNotFoundError(f"Folder not found: {root}")

    files = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in allowed_extensions
    ]
    files.sort(key=lambda p: str(p).lower())
    return files


media_files = collect_media_files(TEMP_ROOT, ALLOWED_EXTENSIONS)
print(f"Found {len(media_files)} media files in {TEMP_ROOT}/ (recursive)")
for p in media_files[:10]:
    print(" -", p)
if len(media_files) > 10:
    print(f"... and {len(media_files) - 10} more")

Found 70 media files in temp/ (recursive)
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\audio_2026-02-19_11-56-27.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-credit2.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-soft-soglasie.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-zaim.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\callcenter.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\collector.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\horeca-bron.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\mfk-reaktvation.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\mnp-client-hold.mp3
 - C:\Users\Alex\whisper_asr_implementation\temp\fromtech\pko-soglasie.mp3
... and 60 more


In [3]:
def enqueue_job(file_path, language, diarization, n_speakers):
    payload = {
        "source": SOURCE_NAME,
        "files": str(file_path.resolve()),
        "type": "mono-stereo",
        "diarization": bool(diarization),
        "language": language,
    }
    if diarization:
        payload["n_speakers"] = int(n_speakers)

    response = requests.post(f"{FRONTMAN_BASE_URL}/add_job", json=payload, timeout=60)
    response.raise_for_status()
    return response.json()


In [4]:
def get_job_status(job_id):
    r = requests.get(f"{FRONTMAN_BASE_URL}/job_status/{job_id}", timeout=30)
    r.raise_for_status()
    return r.json()


def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


results = []
errors = []

for batch_idx, batch_files in enumerate(chunked(media_files, BATCH_SIZE), start=1):
    print(f"\n=== Batch {batch_idx}: enqueue {len(batch_files)} file(s) ===")

    batch_rows = []
    batch_job_ids = []
    for file_path in batch_files:
        try:
            res = enqueue_job(file_path, LANGUAGE, ENABLE_DIARIZATION, N_SPEAKERS)
            row = {
                "file": str(file_path),
                "job_id": res.get("job_id"),
                "queue": res.get("queue"),
                "message": res.get("message"),
                "final_status": None,
                "eta": None,
            }
            results.append(row)
            batch_rows.append(row)
            if row["job_id"] is not None:
                batch_job_ids.append(row["job_id"])
            print(f"queued job_id={row['job_id']} -> {file_path.name}")
        except Exception as exc:
            err = {"file": str(file_path), "error": str(exc), "batch": batch_idx}
            errors.append(err)
            print(f"ERROR enqueue -> {file_path.name}: {exc}")

    if not batch_job_ids:
        print("No jobs enqueued in this batch, moving to next batch.")
        continue

    print(f"Batch {batch_idx}: waiting for {len(batch_job_ids)} job(s) to reach final status...")
    pending = set(batch_job_ids)

    while pending:
        completed_now = []
        for job_id in sorted(pending):
            try:
                status_data = get_job_status(job_id)
                status = status_data.get("status")
                eta = status_data.get("eta")

                for row in batch_rows:
                    if row["job_id"] == job_id:
                        row["final_status"] = status
                        row["eta"] = eta
                        break

                if status in FINAL_STATUSES:
                    completed_now.append(job_id)
            except Exception as exc:
                print(f"job_id={job_id} status check error: {exc}")

        for job_id in completed_now:
            pending.discard(job_id)

        done_count = len(batch_job_ids) - len(pending)
        print(f"Batch {batch_idx} progress: {done_count}/{len(batch_job_ids)} finished")

        if pending:
            time.sleep(POLL_INTERVAL_SEC)

    print(f"Batch {batch_idx}: all queued jobs finished.")


print("\nDone")
print(f"Total queued: {len(results)}")
print(f"Total enqueue errors: {len(errors)}")
if errors:
    print("First errors:")
    print(json.dumps(errors[:5], ensure_ascii=False, indent=2))


=== Batch 1: enqueue 5 file(s) ===
queued job_id=6257 -> audio_2026-02-19_11-56-27.mp3
queued job_id=6258 -> bank-credit2.mp3
queued job_id=6259 -> bank-soft-soglasie.mp3
queued job_id=6260 -> bank-zaim.mp3
queued job_id=6261 -> callcenter.mp3
Batch 1: waiting for 5 job(s) to reach final status...
Batch 1 progress: 0/5 finished
Batch 1 progress: 0/5 finished
Batch 1 progress: 0/5 finished
Batch 1 progress: 0/5 finished
Batch 1 progress: 0/5 finished
Batch 1 progress: 0/5 finished
Batch 1 progress: 2/5 finished
Batch 1 progress: 2/5 finished
Batch 1 progress: 4/5 finished
Batch 1 progress: 4/5 finished
Batch 1 progress: 5/5 finished
Batch 1: all queued jobs finished.

=== Batch 2: enqueue 5 file(s) ===
queued job_id=6262 -> collector.mp3
queued job_id=6263 -> horeca-bron.mp3
queued job_id=6264 -> mfk-reaktvation.mp3
queued job_id=6265 -> mnp-client-hold.mp3
queued job_id=6266 -> pko-soglasie.mp3
Batch 2: waiting for 5 job(s) to reach final status...
Batch 2 progress: 0/5 finished
Batch

In [5]:
# Optional: check current status for enqueued jobs
def get_job_status(job_id):
    r = requests.get(f"{FRONTMAN_BASE_URL}/job_status/{job_id}", timeout=30)
    r.raise_for_status()
    return r.json()


status_preview = []
for row in results[:20]:
    job_id = row["job_id"]
    try:
        status = get_job_status(job_id)
        status_preview.append({"job_id": job_id, **status, "file": row["file"]})
    except Exception as exc:
        status_preview.append({"job_id": job_id, "status": "error", "error": str(exc), "file": row["file"]})

print(json.dumps(status_preview, ensure_ascii=False, indent=2))

[
  {
    "job_id": 6257,
    "status": "transcribed",
    "queue": 0,
    "eta": "2026-05-07 16:09:43",
    "file": "C:\\Users\\Alex\\whisper_asr_implementation\\temp\\fromtech\\audio_2026-02-19_11-56-27.mp3"
  },
  {
    "job_id": 6258,
    "status": "transcribed",
    "queue": 0,
    "eta": "2026-05-07 16:09:50",
    "file": "C:\\Users\\Alex\\whisper_asr_implementation\\temp\\fromtech\\bank-credit2.mp3"
  },
  {
    "job_id": 6259,
    "status": "transcribed",
    "queue": 0,
    "eta": "2026-05-07 16:10:08",
    "file": "C:\\Users\\Alex\\whisper_asr_implementation\\temp\\fromtech\\bank-soft-soglasie.mp3"
  },
  {
    "job_id": 6260,
    "status": "transcribed",
    "queue": 0,
    "eta": "2026-05-07 16:10:08",
    "file": "C:\\Users\\Alex\\whisper_asr_implementation\\temp\\fromtech\\bank-zaim.mp3"
  },
  {
    "job_id": 6261,
    "status": "transcribed",
    "queue": 0,
    "eta": "2026-05-07 16:10:30",
    "file": "C:\\Users\\Alex\\whisper_asr_implementation\\temp\\fromtech\\callc

In [6]:
# Export one TXT sidecar per recording (next to original file)

def get_transcription(job_id):
    r = requests.get(f"{FRONTMAN_BASE_URL}/get_transcription/{job_id}", timeout=60)
    r.raise_for_status()
    return r.json()


def format_ts(seconds):
    # Keep the same style as: 0:33, 1:05, etc.
    sec = max(0, int(round(float(seconds))))
    m = sec // 60
    s = sec % 60
    return f"{m}:{s:02d}"


def bubbles_to_lines(bubbles):
    lines = []
    for item in bubbles:
        if not isinstance(item, dict):
            continue

        start = item.get("start", 0)
        end = item.get("end", 0)
        speaker = item.get("speaker") or item.get("speaker_label") or "SPEAKER_00"
        text = (item.get("text") or "").strip()

        if not text:
            continue

        lines.append(f"{format_ts(start)} - {format_ts(end)} {speaker}:  {text}")
    return lines


saved_txt = []
save_errors = []

for row in results:
    job_id = row.get("job_id")
    audio_path = Path(row.get("file", ""))

    if not job_id:
        save_errors.append({"file": str(audio_path), "error": "missing job_id"})
        continue

    try:
        payload = get_transcription(job_id)
        bubbles = payload.get("transcription", [])

        lines = bubbles_to_lines(bubbles)
        txt_path = audio_path.with_suffix(".txt")

        txt_path.write_text("\n".join(lines), encoding="utf-8")
        saved_txt.append(str(txt_path))
        print(f"saved -> {txt_path}")

    except Exception as exc:
        save_errors.append({"job_id": job_id, "file": str(audio_path), "error": str(exc)})
        print(f"ERROR saving txt for job_id={job_id}: {exc}")

print("\nTXT export done")
print(f"Saved: {len(saved_txt)}")
print(f"Errors: {len(save_errors)}")
if save_errors:
    print(json.dumps(save_errors[:10], ensure_ascii=False, indent=2))

saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\audio_2026-02-19_11-56-27.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-credit2.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-soft-soglasie.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\bank-zaim.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\callcenter.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\collector.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\horeca-bron.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\mfk-reaktvation.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\mnp-client-hold.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\pko-soglasie.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\retail-avito.txt
saved -> C:\Users\Alex\whisper_asr_implementation\temp\fromtech\retail-routing.txt
saved -> C